# Household trip generation: training-only exploratory analysis

The [course assignment](intermediaries/trial_mode/Dontoh%20Project.pdf) requires a common training/test split for regression and cross-classification. This notebook checks the supplied workbook, applies the agreed common eligibility rule, and describes **training households only**. The held-out records are written to a separate workbook before any trip summaries are calculated. Figures and final tables use training data; the test workbook is opened only by model-evaluation code.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
SOURCE = ROOT / 'inputs' / 'households.xlsx'
INTERMEDIATE = ROOT / 'intermediaries'
OUTPUT = ROOT / 'final_outputs'
FIGURES = OUTPUT / 'figures'
TRAIN_BOOK = INTERMEDIATE / 'EDA_training.xlsx'
TEST_BOOK = INTERMEDIATE / 'EDA_heldout_test.xlsx'
FINAL_BOOK = OUTPUT / 'Household_EDA_results.xlsx'
REPORT = OUTPUT / 'EDA_results.md'
for folder in [INTERMEDIATE, OUTPUT, FIGURES]: folder.mkdir(parents=True, exist_ok=True)

FIELDS = ['hnum', 'gta01hhld', 'dwelltype', 'rec-mail', 'n-call', 'tripday',
          'nperson', 'nvehicle', 'nlicence', 'nempft', 'nemppt', 'nemphome',
          'nstudent', 'nhhldtrip', 'expf']
REGIONS = {'Toronto': (1, 481), 'Durham': (501, 765),
           'York': (1001, 1353), 'Peel': (1501, 1753)}
SIZE_TOP, DRIVER_TOP, VEHICLE_TOP = 5, 3, 3
MIN_CELL_N = 30
plt.rcParams.update({'figure.facecolor':'white','axes.spines.top':False,
                     'axes.spines.right':False,'font.size':10})

def write_workbook(path, sheets):
    temporary = path.with_name(path.stem + '_building.xlsx')
    names = [name[:31] for name in sheets]
    assert len(names) == len(set(names))
    with pd.ExcelWriter(temporary, engine='openpyxl') as writer:
        for name, frame in sheets.items():
            frame.to_excel(writer, sheet_name=name[:31], index=False)
            sheet = writer.sheets[name[:31]]
            sheet.freeze_panes = 'A2'
            sheet.auto_filter.ref = sheet.dimensions
    temporary.replace(path)

def grouped_rates(frame, field):
    rows = []
    for level, block in frame.groupby(field, sort=True, observed=True, dropna=False):
        w = block['expf'].to_numpy(float)
        y = block['nhhldtrip'].to_numpy(float)
        rows.append({'variable':field,'level':str(level),'n':len(block),
                     'mean_trips':y.mean(),'weighted_mean_trips':np.average(y,weights=w),
                     'zero_pct':100*np.mean(y==0)})
    return pd.DataFrame(rows)

## 1. Source checks and fixed split

Only named fields B:P are read. The source guide defines vehicle 99 as a numeric count and dwelling 9 as Unknown. The shared modeling eligibility rule removes recorded vehicles ≥10 and unknown dwelling. This is an analytical restriction, recorded in the exclusion sheet. The existing household-ID split is reused so regression and cross-classification evaluate the same households. The source workbook is unchanged.

In [2]:
source = pd.read_excel(SOURCE, sheet_name='2006-hhld', usecols='B:P', dtype=object, engine='openpyxl')
assert source.columns.tolist() == FIELDS
source.insert(0, 'source_excel_row', np.arange(len(source)) + 2)
for field in FIELDS:
    if field != 'rec-mail':
        source[field] = pd.to_numeric(source[field], errors='raise')
        assert np.isfinite(source[field]).all(), field
        if field != 'expf':
            assert source[field].mod(1).eq(0).all(), field
            source[field] = source[field].astype('int64')
assert source['hnum'].is_unique
assert source['dwelltype'].isin([1,2,3,9]).all()
assert source['tripday'].isin([1,2,3,4,5]).all()
assert source['nperson'].between(1,99).all()
assert source['nvehicle'].between(0,99).all()
assert source['nhhldtrip'].between(0,99).all()
assert source['expf'].gt(0).all()
for field in ['nlicence','nempft','nemppt','nemphome','nstudent']:
    assert source[field].between(0,9).all(), field
assert source['nlicence'].le(source['nperson']).all()
assert (source['nempft']+source['nemppt']).le(source['nperson']).all()
assert source['nemphome'].le(source['nempft']+source['nemppt']).all()
assert source['nstudent'].le(source['nperson']).all()

# The saved assignment is the canonical split; the legacy CSV is read only once during migration.
if TRAIN_BOOK.exists():
    split = pd.read_excel(TRAIN_BOOK, sheet_name='Split', dtype={'hnum':'int64'})
else:
    split = pd.read_csv(INTERMEDIATE / 'split_assignment.csv', dtype={'hnum':'int64'})
split = split[['hnum','split']]
assert split['hnum'].is_unique and set(split['split']) == {'train','test'}
assert set(split['hnum']) == set(source['hnum'])
source = source.merge(split, on='hnum', validate='one_to_one', sort=False)
source['exclude_vehicles_10plus'] = source['nvehicle'].ge(10)
source['exclude_unknown_dwelling'] = source['dwelltype'].eq(9)
excluded = source.loc[source['exclude_vehicles_10plus'] | source['exclude_unknown_dwelling']].copy()
excluded['reason'] = [', '.join(reason for flag,reason in
    [(v,'vehicles >=10'),(d,'dwelling 9 Unknown')] if flag) for v,d in
    zip(excluded['exclude_vehicles_10plus'],excluded['exclude_unknown_dwelling'])]
eligible = source.loc[~(source['exclude_vehicles_10plus'] | source['exclude_unknown_dwelling'])]
train = eligible.loc[eligible['split'].eq('train'), ['source_excel_row']+FIELDS+['split']].copy().reset_index(drop=True)
heldout = eligible.loc[eligible['split'].eq('test'), ['source_excel_row']+FIELDS+['split']].copy().reset_index(drop=True)
assert len(train)+len(heldout)+len(excluded)==len(source)
assert set(train['hnum']).isdisjoint(set(heldout['hnum']))

# Store the test records separately, then release them before EDA calculations.
write_workbook(TEST_BOOK, {'Test': heldout})
source_n, train_n, test_n, excluded_n = len(source),len(train),len(heldout),len(excluded)
source_hash = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
del source, eligible, heldout

train['workers'] = train['nempft'] + train['nemppt']
train['dwelling'] = train['dwelltype'].map({1:'House',2:'Apartment',3:'Townhouse'})
train['weekday'] = train['tripday'].map(dict(enumerate(
    ['Monday','Tuesday','Wednesday','Thursday','Friday'],start=1)))
train['region'] = ''
for region,(low,high) in REGIONS.items():
    train.loc[train['gta01hhld'].between(low,high),'region'] = region
assert train['region'].ne('').all()
train['size_band'] = train['nperson'].clip(upper=SIZE_TOP)
train['driver_band'] = train['nlicence'].clip(upper=DRIVER_TOP)
train['vehicle_band'] = train['nvehicle'].clip(upper=VEHICLE_TOP)

quality = pd.DataFrame([
    {'check':'Source schema and numeric ranges','result':'Passed'},
    {'check':'Unique household IDs','result':'Passed'},
    {'check':'Within-household count consistency','result':'Passed'},
    {'check':'Split IDs match source IDs','result':'Passed'},
    {'check':'Train/test IDs disjoint','result':'Passed'},
    {'check':'Source workbook SHA256','result':source_hash},
])
flow = pd.DataFrame([
    {'stage':'Supplied source households','households':source_n},
    {'stage':'Excluded: vehicles >=10','households':int(excluded['exclude_vehicles_10plus'].sum())},
    {'stage':'Excluded: dwelling 9 Unknown','households':int(excluded['exclude_unknown_dwelling'].sum())},
    {'stage':'Unique excluded','households':excluded_n},
    {'stage':'Common eligible training','households':train_n},
    {'stage':'Common eligible held-out test','households':test_n},
])
display(flow)
print('Held-out records saved separately. No test outcomes enter subsequent EDA cells.')

,stage,households
0,Supplied source households,93307
1,Excluded: vehicles >=10,33
2,Excluded: dwelling 9 Unknown,9
3,Unique excluded,42
4,Common eligible training,65283
5,Common eligible held-out test,27982


Held-out records saved separately. No test outcomes enter subsequent EDA cells.


## 2. Training-data distributions and trip rates

The following summaries use the common **training** sample only. Zero-trip households remain. Counts are shown exactly in calculations; top-coded groups are display and cross-classification choices. Weighted means use the household expansion factor; the unweighted and weighted quantities answer different questions.

In [3]:
numeric_fields = ['nperson','nvehicle','nlicence','nempft','nemppt',
                  'nemphome','nstudent','workers','nhhldtrip','expf']
numeric = pd.DataFrame([{'variable':field,'n':len(train),
    'mean':train[field].mean(),'sd':train[field].std(),'median':train[field].median(),
    'p25':train[field].quantile(.25),'p75':train[field].quantile(.75),
    'minimum':train[field].min(),'maximum':train[field].max(),
    'zero_n':int(train[field].eq(0).sum())} for field in numeric_fields])
trip_counts = train['nhhldtrip'].value_counts().sort_index()
trip_distribution = pd.DataFrame({'trips':trip_counts.index,'households':trip_counts.values})
trip_distribution['percent'] = 100*trip_distribution['households']/len(train)
categories = pd.concat([grouped_rates(train,field) for field in
    ['size_band','vehicle_band','driver_band','workers','nstudent','dwelling','weekday','region']],
    ignore_index=True)
display(numeric)
display(categories.head(16))

,variable,n,mean,sd,median,p25,p75,minimum,maximum,zero_n
0,nperson,65283,2.743210,1.422514,2.00,2.00,4.00,1.00,16.00,0
1,nvehicle,65283,1.376775,0.927370,1.00,1.00,2.00,0.00,9.00,10685
2,nlicence,65283,1.732641,0.983905,2.00,1.00,2.00,0.00,9.00,6239
3,nempft,65283,1.111836,0.926965,1.00,0.00,2.00,0.00,7.00,19550
4,nemppt,65283,0.238561,0.513544,0.00,0.00,0.00,0.00,5.00,52078
5,nemphome,65283,0.108221,0.354682,0.00,0.00,0.00,0.00,4.00,59121
6,nstudent,65283,0.636307,0.957027,0.00,0.00,1.00,0.00,9.00,40488
7,workers,65283,1.350398,1.051533,1.00,1.00,2.00,0.00,10.00,16143
8,nhhldtrip,65283,5.652069,4.579926,4.00,2.00,8.00,0.00,46.00,7368
9,expf,65283,19.390925,2.507366,19.37,17.67,20.99,10.83,28.06,0


,variable,level,n,mean_trips,weighted_mean_trips,zero_pct
0,size_band,1,13423,2.026820,2.029926,31.095880
1,size_band,2,20168,4.357200,4.353668,12.866918
2,size_band,3,12227,6.387994,6.375580,2.805267
3,size_band,4,12188,8.313915,8.301515,1.485067
4,size_band,5,7277,10.233063,10.215467,1.030644
5,vehicle_band,0,10685,2.347777,2.357952,34.730931
6,vehicle_band,1,26864,4.854191,4.858143,10.839786
7,vehicle_band,2,21848,7.340901,7.336965,3.066642
8,vehicle_band,3,5886,9.023276,9.038182,1.274210
9,driver_band,0,6239,1.699631,1.714825,46.962654


## 3. Joint household patterns

The principal cross-classification candidate uses household size 1, 2, 3, 4, 5+ and licensed drivers 0, 1, 2, 3+. The workbook records cell support and flags cells below 30 observations. Vehicle-based cells are included as a descriptive alternative. The later model specifies a fallback for sparse or empty cells using training rates only.

In [4]:
def cell_grid(row_field, col_field, row_levels, col_levels):
    grouped = train.groupby([row_field,col_field],observed=True).agg(
        n=('hnum','size'), trips=('nhhldtrip','sum'), weighted_trips=('nhhldtrip',
            lambda x: np.nan), weight_sum=('expf','sum')).reset_index()
    # Weighted trip totals are calculated directly to keep the denominator explicit.
    weighted = train.assign(weighted_trips=train['nhhldtrip']*train['expf']).groupby(
        [row_field,col_field],observed=True)['weighted_trips'].sum().reset_index()
    grouped = grouped.drop(columns='weighted_trips').merge(weighted,on=[row_field,col_field])
    grid = pd.MultiIndex.from_product([row_levels,col_levels],names=[row_field,col_field]).to_frame(index=False)
    result = grid.merge(grouped,on=[row_field,col_field],how='left')
    result['n'] = result['n'].fillna(0).astype(int)
    result['mean_trips'] = result['trips']/result['n'].replace(0,np.nan)
    result['weighted_mean_trips'] = result['weighted_trips']/result['weight_sum']
    result['sparse_under_30'] = result['n'].lt(MIN_CELL_N)
    return result.drop(columns=['trips','weighted_trips'])

size_driver = cell_grid('size_band','driver_band',range(1,SIZE_TOP+1),range(DRIVER_TOP+1))
size_driver['structurally_possible'] = size_driver['size_band'].eq(SIZE_TOP) | size_driver['driver_band'].le(size_driver['size_band'])
size_vehicle = cell_grid('size_band','vehicle_band',range(1,SIZE_TOP+1),range(VEHICLE_TOP+1))
assert size_driver['n'].sum() == len(train) == size_vehicle['n'].sum()
display(size_driver)

,size_band,driver_band,n,weight_sum,mean_trips,weighted_mean_trips,sparse_under_30,structurally_possible
0,1,0,4073,78814.92,1.067272,1.073540,False,True
1,1,1,9350,181832.04,2.444813,2.444471,False,True
2,1,2,0,NaN,NaN,NaN,True,False
3,1,3,0,NaN,NaN,NaN,True,False
4,2,0,1460,28449.58,2.110274,2.135256,False,True
5,2,1,5462,106267.29,3.798425,3.803781,False,True
6,2,2,13246,255248.07,4.835271,4.829863,False,True
7,2,3,0,NaN,NaN,NaN,True,False
8,3,0,403,7939.05,3.885856,3.894375,False,True
9,3,1,2208,42985.86,5.471920,5.462194,False,True


## 4. Region, zone support, correlations, and figures

These are descriptive checks for interpretation and model design. They do not use the reserved test set. The plots are exported separately so they can be placed in the paper without copying spreadsheet cells.

In [5]:
zone_support = train.groupby('gta01hhld').agg(
    households=('hnum','size'),observed_sample_trips=('nhhldtrip','sum'),
    expansion_factor_sum=('expf','sum')).reset_index()
correlation_fields = ['nperson','nvehicle','nlicence','nempft','nemppt','nemphome','nstudent','workers','nhhldtrip']
correlations = train[correlation_fields].corr().reset_index().rename(columns={'index':'variable'})
weighting = pd.DataFrame([
    {'measure':'Training household mean trips','unweighted':train['nhhldtrip'].mean(),
     'expansion_weighted':np.average(train['nhhldtrip'],weights=train['expf'])},
    {'measure':'Training zero-trip share','unweighted':train['nhhldtrip'].eq(0).mean(),
     'expansion_weighted':np.average(train['nhhldtrip'].eq(0),weights=train['expf'])},
])

fig,ax=plt.subplots(figsize=(9,4.5))
ax.bar(train['nhhldtrip'].clip(upper=25).value_counts().sort_index().index,
       train['nhhldtrip'].clip(upper=25).value_counts().sort_index().values,color='#2F80ED')
ax.set(xlabel='Recorded weekday household trips (25+ grouped)',ylabel='Training households',
       title='Training-set trip distribution')
fig.tight_layout();fig.savefig(FIGURES/'EDA_01_trip_distribution.png',dpi=220);plt.close(fig)

fig,axes=plt.subplots(1,3,figsize=(12,3.8))
for ax,field,title in zip(axes,['size_band','vehicle_band','driver_band'],
                          ['Household size','Available vehicles','Licensed drivers']):
    block=categories.loc[categories['variable'].eq(field)]
    ax.plot(block['level'],block['mean_trips'],marker='o',color='#2F80ED')
    ax.set(xlabel=title,ylabel='Mean trips',title=title)
fig.tight_layout();fig.savefig(FIGURES/'EDA_02_group_rates.png',dpi=220);plt.close(fig)

heat=size_driver.pivot(index='size_band',columns='driver_band',values='mean_trips')
fig,ax=plt.subplots(figsize=(7,4.5));im=ax.imshow(heat,cmap='Blues',aspect='auto')
ax.set(xticks=range(len(heat.columns)),xticklabels=[str(x) for x in heat.columns],
       yticks=range(len(heat.index)),yticklabels=[str(x) for x in heat.index],
       xlabel='Licensed drivers (3+ grouped)',ylabel='Household size (5+ grouped)',
       title='Training mean trips by size and drivers')
for r in range(len(heat.index)):
    for c in range(len(heat.columns)):
        if np.isfinite(heat.iloc[r,c]):ax.text(c,r,f'{heat.iloc[r,c]:.1f}',ha='center',va='center')
fig.colorbar(im,ax=ax,label='Mean trips');fig.tight_layout()
fig.savefig(FIGURES/'EDA_03_size_driver_cells.png',dpi=220);plt.close(fig)
display(weighting)

,measure,unweighted,expansion_weighted
0,Training household mean trips,5.652069,5.648023
1,Training zero-trip share,0.112862,0.112077


## 5. Detailed descriptive tables and restored figures

The following profiles recover the original EDA's exact and grouped household rates, sample support, zero-trip shares, trip frequencies, correlation views, regional cells, and adjacent cross-classification differences. Model-facing statistics and figures use eligible **training households only**. Two vehicle review tables and figures use training-assigned households before exclusions; no held-out test households enter those reviews. Display caps group tails for legibility; exact values remain available in the detailed tables. Figures are descriptive and do not establish causal effects.

In [6]:
# Detailed profiles of the same eligible training sample.
count_fields = ['nperson','nvehicle','nlicence','nempft','nemppt','nemphome','nstudent','workers']
category_fields = ['dwelling','weekday','region']
display_caps = {'nperson':5,'nvehicle':3,'nlicence':3,'nempft':4,'nemppt':3,
                'nemphome':3,'nstudent':4,'workers':4}
detail_tables = {}

def rate_table(frame, field, grouped=False):
    values = frame[field].clip(upper=display_caps[field]) if grouped and field in display_caps else frame[field]
    data = frame.assign(_level=values, _weighted_trips=frame['nhhldtrip']*frame['expf'],
                        _weighted_zero=frame['nhhldtrip'].eq(0)*frame['expf'])
    out = data.groupby('_level', sort=True, observed=True).agg(
        n=('hnum','size'), mean_trips=('nhhldtrip','mean'), sd_trips=('nhhldtrip','std'),
        median_trips=('nhhldtrip','median'), zero_n=('nhhldtrip',lambda s: int(s.eq(0).sum())),
        weight_sum=('expf','sum'), weighted_trips=('_weighted_trips','sum'),
        weighted_zero=('_weighted_zero','sum')).reset_index().rename(columns={'_level':'level'})
    out['zero_pct'] = 100*out['zero_n']/out['n']
    out['weighted_mean_trips'] = out['weighted_trips']/out['weight_sum']
    out['weighted_zero_pct'] = 100*out['weighted_zero']/out['weight_sum']
    if field in category_fields:
        order = {'dwelling':['House','Apartment','Townhouse'],
                 'weekday':['Monday','Tuesday','Wednesday','Thursday','Friday'],
                 'region':['Toronto','Durham','York','Peel']}[field]
        out['_sort'] = out['level'].map({level:i for i,level in enumerate(order)})
        out = out.sort_values('_sort').drop(columns='_sort')
    out['variable'] = field
    out['display_grouped'] = grouped and field in display_caps
    out['level_label'] = out['level'].astype(str)
    if grouped and field in display_caps:
        out.loc[out['level'].eq(display_caps[field]),'level_label'] += '+'
    return out[['variable','level','level_label','display_grouped','n','mean_trips',
                'sd_trips','median_trips','zero_n','zero_pct','weight_sum',
                'weighted_mean_trips','weighted_zero_pct']]

detail_tables['ExactGroupRates'] = pd.concat(
    [rate_table(train,f) for f in count_fields+category_fields],ignore_index=True)
detail_tables['DisplayGroupRates'] = pd.concat(
    [rate_table(train,f,True) for f in count_fields+category_fields],ignore_index=True)
detail_tables['CategoricalFrequencies'] = pd.concat([
    train.groupby(field,observed=True).agg(n=('hnum','size'),weight_sum=('expf','sum'))
         .reset_index().rename(columns={field:'level'}).assign(variable=field)
    for field in category_fields+['size_band','vehicle_band','driver_band']],ignore_index=True)
detail_tables['CategoricalFrequencies']['sample_pct'] = (
    100*detail_tables['CategoricalFrequencies']['n']/len(train))

detail_tables['NumericDetail'] = pd.DataFrame([{
    'variable':f,'n':len(train),'mean':train[f].mean(),'sd':train[f].std(),
    'se_mean':train[f].sem(),'coefficient_of_variation':train[f].std()/train[f].mean() if train[f].mean() else np.nan,
    'minimum':train[f].min(),'p05':train[f].quantile(.05),'p10':train[f].quantile(.10),
    'p25':train[f].quantile(.25),'median':train[f].median(),'p75':train[f].quantile(.75),
    'p90':train[f].quantile(.90),'p95':train[f].quantile(.95),'p99':train[f].quantile(.99),
    'maximum':train[f].max(),'zero_n':int(train[f].eq(0).sum()),
    'zero_pct':100*train[f].eq(0).mean(),'missing_n':int(train[f].isna().sum())}
    for f in count_fields+['nhhldtrip','expf']])

def cross_cells(frame, col_field, cap):
    data = frame.assign(size=frame['nperson'].clip(upper=SIZE_TOP),
                        level=frame[col_field].clip(upper=cap),
                        weighted_trips=frame['nhhldtrip']*frame['expf'])
    cells = data.groupby(['size','level'],observed=True).agg(
        n=('hnum','size'),mean_trips=('nhhldtrip','mean'),sd_trips=('nhhldtrip','std'),
        zero_n=('nhhldtrip',lambda s: int(s.eq(0).sum())),
        weight_sum=('expf','sum'),weighted_trips=('weighted_trips','sum')).reset_index()
    grid = pd.MultiIndex.from_product([range(1,SIZE_TOP+1),range(cap+1)],
                                      names=['size','level']).to_frame(index=False)
    cells = grid.merge(cells,on=['size','level'],how='left')
    cells[['n','zero_n']] = cells[['n','zero_n']].fillna(0).astype(int)
    cells['weight_sum'] = cells['weight_sum'].fillna(0)
    cells['weighted_mean_trips'] = cells['weighted_trips']/cells['weight_sum'].replace(0,np.nan)
    cells['zero_pct'] = 100*cells['zero_n']/cells['n'].replace(0,np.nan)
    cells['sparse_under_30'] = cells['n'].lt(MIN_CELL_N)
    cells['scheme'] = f'5+ persons by {cap}+ {col_field}'
    cells['structurally_possible'] = True if col_field=='nvehicle' else (
        cells['size'].eq(SIZE_TOP) | cells['level'].le(cells['size']))
    return cells.drop(columns='weighted_trips')

detail_tables['VehicleCellsAlt'] = cross_cells(train,'nvehicle',4)
detail_tables['DriverCellsAlt'] = cross_cells(train,'nlicence',4)
all_cells = pd.concat([
    size_vehicle.rename(columns={'size_band':'size','vehicle_band':'level'}).assign(scheme='5+ persons by 3+ nvehicle'),
    detail_tables['VehicleCellsAlt'],
    size_driver.rename(columns={'size_band':'size','driver_band':'level'}).assign(scheme='5+ persons by 3+ nlicence'),
    detail_tables['DriverCellsAlt']],ignore_index=True)
adjacent = []
for scheme,block in all_cells.groupby('scheme',sort=False):
    for change,fixed in [('size','level'),('level','size')]:
        for fixed_level,line in block.groupby(fixed):
            line = line.sort_values(change)
            for (_,left),(_,right) in zip(line.iloc[:-1].iterrows(),line.iloc[1:].iterrows()):
                adjacent.append({'scheme':scheme,'changing_dimension':change,
                    'fixed_dimension':fixed,'fixed_level':fixed_level,
                    'from_level':int(left[change]),'to_level':int(right[change]),
                    'n_from':int(left['n']),'n_to':int(right['n']),
                    'mean_difference':right['mean_trips']-left['mean_trips'],
                    'weighted_mean_difference':right['weighted_mean_trips']-left['weighted_mean_trips'],
                    'both_supported_30':min(left['n'],right['n'])>=MIN_CELL_N})
detail_tables['AdjacentCellDifferences'] = pd.DataFrame(adjacent)

frequency = []
for field,cap in [('nperson',5),('nvehicle',3)]:
    data = train.assign(level=train[field].clip(upper=cap),trips=train['nhhldtrip'].clip(upper=20))
    counts = data.groupby(['level','trips']).size().rename('n')
    grid = pd.MultiIndex.from_product([range(1 if field=='nperson' else 0,cap+1),
                                       range(21)],names=['level','trips'])
    block = counts.reindex(grid,fill_value=0).reset_index()
    block['variable']=field
    block['level_label']=block['level'].astype(str)
    block.loc[block['level'].eq(cap),'level_label']+='+'
    block['trip_label']=block['trips'].astype(str)
    block.loc[block['trips'].eq(20),'trip_label']='20+'
    block['group_n']=block.groupby('level')['n'].transform('sum')
    frequency.append(block)
detail_tables['TripFrequencies'] = pd.concat(frequency,ignore_index=True)
assert all(detail_tables['TripFrequencies'].groupby('variable')['n'].sum().eq(len(train)))

predictors = count_fields
pair = []
for left in predictors+['nhhldtrip']:
    for right in predictors+['nhhldtrip']:
        pair.append({'variable_1':left,'variable_2':right,'n':len(train),
                     'pearson':train[left].corr(train[right],method='pearson'),
                     'spearman':train[left].corr(train[right],method='spearman')})
detail_tables['PairwiseCorrelations'] = pd.DataFrame(pair)
detail_tables['OutcomeCorrelations'] = detail_tables['PairwiseCorrelations'].loc[
    detail_tables['PairwiseCorrelations']['variable_2'].eq('nhhldtrip') &
    detail_tables['PairwiseCorrelations']['variable_1'].ne('nhhldtrip')].copy()

regional = []
for region,block in train.groupby('region',sort=True):
    cells = cross_cells(block,'nlicence',3)
    cells.insert(0,'region',region)
    cells['region_n']=len(block)
    cells['share_of_region_pct']=100*cells['n']/len(block)
    regional.append(cells)
detail_tables['RegionalDriverCells'] = pd.concat(regional,ignore_index=True)

# Pre-exclusion review uses training-assigned households only; test rows stay hidden.
pre_exclusion_train = pd.concat([train[['hnum','nvehicle','nperson','nlicence','dwelltype','nhhldtrip']],
    excluded.loc[excluded['split'].eq('train'),
                 ['hnum','nvehicle','nperson','nlicence','dwelltype','nhhldtrip']]],ignore_index=True)
assert pre_exclusion_train['hnum'].is_unique
review_counts = pre_exclusion_train['nvehicle'].value_counts().sort_index()
detail_tables['VehiclePreExclusion'] = pd.DataFrame({
    'nvehicle':review_counts.index,'training_households':review_counts.values})
detail_tables['VehiclePreExclusion']['training_share_pct'] = (
    100*detail_tables['VehiclePreExclusion']['training_households']/len(pre_exclusion_train))
vehicle_tail = pre_exclusion_train.loc[pre_exclusion_train['nvehicle'].ge(5)].copy()
vehicle_tail['vehicle_band'] = np.where(vehicle_tail['nvehicle'].ge(10),'10+','5-9')
detail_tables['VehicleTailReview'] = vehicle_tail.groupby('vehicle_band').agg(
    n=('hnum','size'),mean_size=('nperson','mean'),mean_licensed=('nlicence','mean'),
    mean_trips=('nhhldtrip','mean'),median_trips=('nhhldtrip','median'),
    house_n=('dwelltype',lambda v:int(v.eq(1).sum())),
    apartment_n=('dwelltype',lambda v:int(v.eq(2).sum())),
    townhouse_n=('dwelltype',lambda v:int(v.eq(3).sum())),
    unknown_dwelling_n=('dwelltype',lambda v:int(v.eq(9).sum()))).reset_index()

display(detail_tables['NumericDetail'])
display(detail_tables['ExactGroupRates'].head(12))
display(detail_tables['OutcomeCorrelations'])
print(f"Prepared {len(detail_tables)} additional training-only descriptive tables.")

,variable,n,mean,sd,se_mean,coefficient_of_variation,minimum,p05,p10,p25,median,p75,p90,p95,p99,maximum,zero_n,zero_pct,missing_n
0,nperson,65283,2.743210,1.422514,0.005567,0.518558,1.00,1.00,1.00,2.00,2.00,4.00,5.00,5.00,7.00,16.00,0,0.000000,0
1,nvehicle,65283,1.376775,0.927370,0.003630,0.673581,0.00,0.00,0.00,1.00,1.00,2.00,2.00,3.00,4.00,9.00,10685,16.367201,0
2,nlicence,65283,1.732641,0.983905,0.003851,0.567864,0.00,0.00,1.00,1.00,2.00,2.00,3.00,3.00,4.00,9.00,6239,9.556852,0
3,nempft,65283,1.111836,0.926965,0.003628,0.833724,0.00,0.00,0.00,0.00,1.00,2.00,2.00,3.00,4.00,7.00,19550,29.946540,0
4,nemppt,65283,0.238561,0.513544,0.002010,2.152670,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,2.00,5.00,52078,79.772682,0
5,nemphome,65283,0.108221,0.354682,0.001388,3.277379,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,2.00,4.00,59121,90.561096,0
6,nstudent,65283,0.636307,0.957027,0.003746,1.504034,0.00,0.00,0.00,0.00,0.00,1.00,2.00,3.00,4.00,9.00,40488,62.019209,0
7,workers,65283,1.350398,1.051533,0.004116,0.778684,0.00,0.00,0.00,1.00,1.00,2.00,3.00,3.00,4.00,10.00,16143,24.727724,0
8,nhhldtrip,65283,5.652069,4.579926,0.017925,0.810310,0.00,0.00,0.00,2.00,4.00,8.00,12.00,14.00,21.00,46.00,7368,11.286246,0
9,expf,65283,19.390925,2.507366,0.009813,0.129306,10.83,15.32,16.33,17.67,19.37,20.99,22.54,23.53,26.13,28.06,0,0.000000,0


,variable,level,level_label,display_grouped,n,mean_trips,sd_trips,median_trips,zero_n,zero_pct,weight_sum,weighted_mean_trips,weighted_zero_pct
0,nperson,1,1,False,13423,2.026820,1.837099,2.0,4174,31.095880,260646.96,2.029926,30.873723
1,nperson,2,2,False,20168,4.357200,3.031043,4.0,2595,12.866918,389964.94,4.353668,12.757796
2,nperson,3,3,False,12227,6.387994,3.634645,6.0,343,2.805267,236891.46,6.375580,2.801173
3,nperson,4,4,False,12188,8.313915,4.701624,8.0,181,1.485067,236603.56,8.301515,1.494965
4,nperson,5,5,False,4906,9.845495,5.644820,9.0,58,1.182226,95516.24,9.832555,1.213710
5,nperson,6,6,False,1657,10.563066,5.891412,10.0,12,0.724200,32343.22,10.528595,0.717956
6,nperson,7,7,False,459,11.396514,6.467560,10.0,1,0.217865,8938.85,11.376670,0.191524
7,nperson,8,8,False,158,12.208861,7.140138,10.0,3,1.898734,3091.49,12.193402,1.738967
8,nperson,9,9,False,55,13.981818,6.507804,14.0,1,1.818182,1087.06,13.914724,1.854543
9,nperson,10,10,False,27,17.629630,10.307384,15.0,0,0.000000,530.33,17.729470,0.000000


,variable_1,variable_2,n,pearson,spearman
8,nperson,nhhldtrip,65283,0.572976,0.614265
17,nvehicle,nhhldtrip,65283,0.429742,0.481009
26,nlicence,nhhldtrip,65283,0.514321,0.558031
35,nempft,nhhldtrip,65283,0.440596,0.507300
44,nemppt,nhhldtrip,65283,0.292483,0.262839
53,nemphome,nhhldtrip,65283,0.086776,0.081178
62,nstudent,nhhldtrip,65283,0.519875,0.505847
71,workers,nhhldtrip,65283,0.531244,0.580124


Prepared 13 additional training-only descriptive tables.


In [7]:
# Restored figure series from the original EDA, recalculated on training households.
def save_eda_figure(fig, number):
    fig.savefig(FIGURES/f'Figure_ED{number:02d}.png',dpi=240,facecolor='white',bbox_inches='tight')
    plt.close(fig)

def plot_rates(fields, number, ncols=2):
    nrows=int(np.ceil(len(fields)/ncols))
    fig,axes=plt.subplots(nrows,ncols,figsize=(5.2*ncols,3.5*nrows),squeeze=False)
    for ax,field in zip(axes.flat,fields):
        data=detail_tables['DisplayGroupRates'].query('variable == @field')
        x=np.arange(len(data))
        ax.plot(x,data['mean_trips'],marker='o',color='#2673b8')
        ax.set_xticks(x,data['level_label'].astype(str),rotation=35 if field in category_fields else 0)
        ax.set(xlabel=field,ylabel='Mean trips per household',title=f'{field} (training)')
        for x0,y,n in zip(x,data['mean_trips'],data['n']):
            ax.annotate(f'n={n:,}',(x0,y),xytext=(0,7),textcoords='offset points',
                        ha='center',fontsize=7,rotation=35)
    for ax in axes.flat[len(fields):]:ax.axis('off')
    fig.tight_layout();save_eda_figure(fig,number)

def plot_cell_heat(cells,number,title):
    fig,axes=plt.subplots(1,2,figsize=(11,4.5))
    for ax,value,label,cmap in zip(axes,['mean_trips','n'],['Mean trips','Households'],['Blues','Greens']):
        matrix=cells.pivot(index='size',columns='level',values=value)
        image=ax.imshow(matrix,cmap=cmap,aspect='auto')
        ax.set(xticks=range(len(matrix.columns)),xticklabels=[str(v) for v in matrix.columns],
               yticks=range(len(matrix.index)),yticklabels=[str(v) for v in matrix.index],
               xlabel='Grouped level',ylabel='Household size (5+ grouped)',title=f'{title}: {label}')
        for r in range(len(matrix.index)):
            for c in range(len(matrix.columns)):
                val=matrix.iloc[r,c]
                if np.isfinite(val):ax.text(c,r,f'{val:,.1f}' if value=='mean_trips' else f'{val:,.0f}',
                                            ha='center',va='center',fontsize=8)
        fig.colorbar(image,ax=ax,shrink=.8)
    fig.tight_layout();save_eda_figure(fig,number)

# ED01: trip-count distribution.
fig,ax=plt.subplots(figsize=(10,4.2))
counts=train['nhhldtrip'].clip(upper=25).value_counts().sort_index()
ax.bar(counts.index,counts.values,color='#2673b8')
ax.set(xlabel='Recorded weekday trips (25+ grouped)',ylabel='Training households',
       title='Training household trip distribution')
fig.tight_layout();save_eda_figure(fig,1)

# ED02: eight predictor distributions.
fig,axes=plt.subplots(2,4,figsize=(14,6.8))
for ax,field in zip(axes.flat,count_fields):
    cap=display_caps[field]
    counts=train[field].clip(upper=cap).value_counts().sort_index()
    ax.bar(counts.index.astype(str),counts.values,color='#579e80')
    labels=[f'{v}+' if v==cap else str(v) for v in counts.index]
    ax.set_xticks(range(len(labels)),labels)
    ax.set(title=field,ylabel='Training households')
fig.tight_layout();save_eda_figure(fig,2)

plot_rates(['nperson','nvehicle','nlicence','workers'],3)
plot_cell_heat(size_vehicle.rename(columns={'size_band':'size','vehicle_band':'level'}),4,
               'Persons by vehicles (training)')
plot_rates(['nempft','nemppt','nemphome','nstudent'],5)

# ED06: predictor rank correlations.
matrix=train[count_fields].corr(method='spearman')
fig,ax=plt.subplots(figsize=(8,6.6));im=ax.imshow(matrix,vmin=-1,vmax=1,cmap='coolwarm')
ax.set(xticks=range(len(count_fields)),xticklabels=count_fields,
       yticks=range(len(count_fields)),yticklabels=count_fields,
       title='Training predictor Spearman correlations')
plt.setp(ax.get_xticklabels(),rotation=45,ha='right')
for r in range(len(count_fields)):
    for c in range(len(count_fields)):
        ax.text(c,r,f'{matrix.iloc[r,c]:.2f}',ha='center',va='center',fontsize=7)
fig.colorbar(im,ax=ax,label='Spearman correlation',shrink=.8)
fig.tight_layout();save_eda_figure(fig,6)

# ED07: joint trip frequency by size and vehicle group.
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for ax,field in zip(axes,['nperson','nvehicle']):
    block=detail_tables['TripFrequencies'].query('variable == @field')
    matrix=block.pivot(index='level',columns='trips',values='n')
    im=ax.imshow(np.log1p(matrix),aspect='auto',cmap='Blues')
    ax.set(yticks=range(len(matrix.index)),yticklabels=matrix.index,
           xlabel='Trips (20+ grouped)',ylabel=field,title=f'{field} × household trips (training)')
    ax.set_xticks([0,5,10,15,20],['0','5','10','15','20+'])
    fig.colorbar(im,ax=ax,label='log(1 + households)',shrink=.8)
fig.tight_layout();save_eda_figure(fig,7)

# ED08: outcome correlations in two metrics.
data=detail_tables['OutcomeCorrelations'].set_index('variable_1').reindex(count_fields)
fig,axes=plt.subplots(1,2,figsize=(11,4.4),sharey=True)
for ax,metric in zip(axes,['pearson','spearman']):
    ax.barh(data.index,data[metric],color='#2673b8')
    ax.axvline(0,color='black',lw=.7)
    ax.set(xlabel=f'{metric.title()} correlation with trips',title=f'{metric.title()} (training)')
fig.tight_layout();save_eda_figure(fig,8)

plot_rates(['dwelling','weekday','region'],9,ncols=3)
plot_cell_heat(size_driver.rename(columns={'size_band':'size','driver_band':'level'}),10,
               'Persons by licensed drivers (training)')

# ED11: driver profiles within household-size groups, with cell support.
fig,ax=plt.subplots(figsize=(9,5))
for size,block in size_driver.groupby('size_band'):
    ax.plot(block['driver_band'],block['mean_trips'],marker='o',label=f'{size}{"+" if size==SIZE_TOP else ""} persons')
    for _,row in block.iterrows():
        if row['n'] and row['n']<MIN_CELL_N:
            ax.annotate('sparse',(row['driver_band'],row['mean_trips']),fontsize=7)
ax.set(xticks=range(DRIVER_TOP+1),xticklabels=['0','1','2','3+'],
       xlabel='Licensed drivers',ylabel='Mean trips per household',
       title='Training trip rates by household size and licensed drivers')
ax.legend(title='Household size',fontsize=8)
fig.tight_layout();save_eda_figure(fig,11)
# S1: raw vehicle frequency review before eligibility screening, training assignment only.
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
counts=detail_tables['VehiclePreExclusion'].set_index('nvehicle')['training_households']
main=[int(counts.get(k,0)) for k in range(10)]+[int(counts.loc[counts.index>=10].sum())]
axes[0].bar(range(11),main,color='#2673b8')
axes[0].set(xticks=range(11),xticklabels=[str(k) for k in range(10)]+['10+'],
            xlabel='Vehicles available',ylabel='Training-assigned households',
            title='Recorded vehicle counts before exclusions')
tail=counts.loc[counts.index>=5]
axes[1].bar(tail.index.astype(str),tail.values,color='#d48345')
axes[1].tick_params(axis='x',labelrotation=60)
axes[1].set(xlabel='Exact recorded vehicles',ylabel='Training-assigned households',
            title='Vehicle tail before exclusions')
fig.tight_layout()
fig.savefig(FIGURES/'S1_vehicle_frequencies.png',dpi=240,facecolor='white',bbox_inches='tight')
plt.close(fig)

# RV01: companion characteristics of 5+ vehicle households before exclusions.
fig,axes=plt.subplots(2,2,figsize=(11,7.5))
for ax,field,title in zip(axes.flat[:3],['nperson','nlicence','nhhldtrip'],
                          ['Household size','Licensed drivers','Recorded weekday trips']):
    for band,color in [('5-9','#2673b8'),('10+','#d48345')]:
        block=vehicle_tail.loc[vehicle_tail['vehicle_band'].eq(band)]
        ax.scatter(block['nvehicle'],block[field],s=24,alpha=.65,color=color,
                   label=f'{band} vehicles (n={len(block)})')
    ax.set(xscale='log',xlabel='Recorded vehicles (log scale)',ylabel=title,title=title)
    ax.set_xticks([5,10,20,40,99],['5','10','20','40','99'])
axes[0,0].legend(fontsize=8)
for band,color in [('5-9','#2673b8'),('10+','#d48345')]:
    block=vehicle_tail.loc[vehicle_tail['vehicle_band'].eq(band)]
    shares=block['dwelltype'].value_counts(normalize=True).reindex([1,2,3,9],fill_value=0)
    axes[1,1].bar(np.arange(4)+(-.18 if band=='5-9' else .18),100*shares,
                  width=.35,label=band,color=color)
axes[1,1].set(xticks=range(4),xticklabels=['House','Apartment','Townhouse','Unknown'],
              ylabel='Within-band share (%)',title='Dwelling type')
axes[1,1].legend(fontsize=8)
fig.suptitle('Training-assigned households with 5+ vehicles before exclusions')
fig.tight_layout()
fig.savefig(FIGURES/'Figure_RV01_vehicle_household_review.png',dpi=240,facecolor='white',bbox_inches='tight')
plt.close(fig)
print('Saved 13 restored training-only figures plus 3 current EDA figures.')

Saved 13 restored training-only figures plus 3 current EDA figures.


## 6. Final outputs and reproducibility

One workbook contains the final training-only EDA tables. The intermediate training workbook contains the split assignment, eligible training records, and exclusion ledger. The held-out records are isolated in their own workbook. The regression and cross-classification scripts may open the held-out workbook only after they have fixed their models using training data.

In [8]:
dictionary = pd.DataFrame([
    {'field':'nhhldtrip','meaning':'Recorded household trips on survey weekday; outcome covers ages 11+'},
    {'field':'nperson','meaning':'Household residents; includes ages below trip-recording threshold'},
    {'field':'nvehicle','meaning':'Numeric vehicles available; 99 is documented as numeric in source guide'},
    {'field':'nlicence','meaning':'Licensed drivers in household'},
    {'field':'nempft / nemppt','meaning':'Full-time / part-time workers; work-at-home persons already included'},
    {'field':'nstudent','meaning':'Students; children aged 6-10 are assumed full-time students'},
    {'field':'dwelltype','meaning':'1 House, 2 Apartment, 3 Townhouse, 9 Unknown'},
    {'field':'tripday','meaning':'Survey weekday, Monday through Friday'},
    {'field':'expf','meaning':'Household expansion factor; not a substantive predictor'},
    {'field':'gta01hhld','meaning':'Residential traffic zone (2001 zone numbering)'},
])
final_tables = {
    'SampleFlow':flow,'DataDictionary':dictionary,'NumericSummary':numeric,
    'TripDistribution':trip_distribution,'GroupTripRates':categories,
    'SizeDriverCells':size_driver,'SizeVehicleCells':size_vehicle,
    'ZoneSupport':zone_support,'Correlations':correlations,'Weighting':weighting,
    **detail_tables,
}
contents = pd.DataFrame([{'sheet':name,'rows':len(frame),'columns':len(frame.columns)}
                         for name,frame in final_tables.items()])
write_workbook(FINAL_BOOK,{'Contents':contents,**final_tables})
write_workbook(TRAIN_BOOK,{'Split':split,'Train':train,'Exclusions':excluded,'QualityChecks':quality})

trip_mean = train['nhhldtrip'].mean()
zero_n = int(train['nhhldtrip'].eq(0).sum())
REPORT.write_text(f'''# Training-only household EDA

The 2006 household source contains {source_n:,} records. Applying the common vehicle >=10 and unknown-dwelling exclusions leaves {train_n:,} training households and {test_n:,} held-out households. {excluded_n} records are excluded from both methods; reasons are in the intermediate workbook. The source workbook is unchanged.

All trip summaries, group rates, cross-classification cells, correlations, and zone support in [Household_EDA_results.xlsx](Household_EDA_results.xlsx) use training records only. Mean recorded trips are {trip_mean:.3f}; {zero_n:,} training households report zero trips. The test records are stored in `intermediaries/EDA_heldout_test.xlsx` and were not read for EDA. The common split and training records are in `intermediaries/EDA_training.xlsx`.

The workbook includes exact and grouped rates, detailed quantiles, zero-trip shares, joint trip frequencies, regional cells, and Pearson and Spearman correlations. Thirteen restored figures (`figures/Figure_ED01.png` through `figures/Figure_ED11.png`, `figures/S1_vehicle_frequencies.png`, and `figures/Figure_RV01_vehicle_household_review.png`) are available. The two vehicle review figures use training-assigned households before exclusions; all other figures use eligible training households. Display caps affect charts and grouped views only.\n\nThe proposed cross-classification uses household size 1/2/3/4/5+ and licensed drivers 0/1/2/3+, with sparse cells flagged below 30 training records. The cross-classification model specifies its fallback rates from training data before evaluating the test set. `expf` is retained for expanded totals, while household-level unweighted results remain explicit.

The source [data guide](../inputs/dataguide2006_v1.pdf) defines vehicle 99 as numeric, dwelling 9 as Unknown, and trips as recorded for household members aged 11+. The [assignment](../intermediaries/trial_mode/Dontoh%20Project.pdf) requires a common held-out test for regression and cross-classification.

Source workbook SHA256: {source_hash}
''',encoding='utf-8')

assert FINAL_BOOK.exists() and TRAIN_BOOK.exists() and TEST_BOOK.exists()
assert contents['rows'].sum()>0 and size_driver['n'].sum()==train_n
print(f'Saved {len(final_tables)} final EDA tables, 16 figures, and separate training/test workbooks.')
display(contents)

Saved 23 final EDA tables, 16 figures, and separate training/test workbooks.


,sheet,rows,columns
0,SampleFlow,6,2
1,DataDictionary,10,2
2,NumericSummary,10,10
3,TripDistribution,47,3
4,GroupTripRates,45,6
5,SizeDriverCells,20,8
6,SizeVehicleCells,20,7
7,ZoneSupport,1156,4
8,Correlations,9,10
9,Weighting,2,3
